# Plant Disease Models v2 (real-world training)

**How to run**
1. `Runtime > Change runtime type > T4 GPU`
2. `Runtime > Run all`
3. Colab asks once for Google Drive access (so finished models survive a disconnect). Click **Allow**.
4. Wait about 2 hours. A zip downloads at the end and is also saved to `MyDrive/plant_models_v2`.

**What it does**
- Classifier: PlantVillage + real field photos (PlantDoc leaf crops) + an `unknown` class (non-plant photos and textures).
- Splits PlantVillage by physical leaf, so near-duplicate shots never leak between train and validation.
- Judges the model only on real-world photos it never saw.
- Heavier augmentation, including pasting lab leaves onto random backgrounds.
- Two-step fine-tuning (head first, then full network at a low learning rate) and confidence calibration.
- Detector: YOLO11n trained on PlantDoc leaf boxes, one class `leaf`.

**Optional:** to test on your own photos, upload them to `/content/my_test_photos/<class name>/` before running
(class names as in PlantVillage, e.g. `Tomato___Early_blight`, or `unknown`). Skipped if the folder is missing.

In [ ]:
SAVE_TO_DRIVE = True
SEED = 42
CLS_EPOCH_SIZE = 25000
CLS_HEAD_EPOCHS = 2
CLS_FULL_EPOCHS = 12
CLS_BATCH = 64
YOLO_EPOCHS = 60

!nvidia-smi -L
!pip install -q ultralytics timm onnx onnxruntime onnxslim onnxscript

import os, random, re, shutil, subprocess, tarfile, urllib.request, zlib, json
from pathlib import Path

random.seed(SEED)
DATA = Path("/content/data")
DATA.mkdir(parents=True, exist_ok=True)

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/plant_models_v2")
else:
    OUT = Path("/content/plant_models_v2")
EXPORT = OUT / "export"
FIGURES = OUT / "figures"
for d in (EXPORT, FIGURES):
    d.mkdir(parents=True, exist_ok=True)


def run(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)


def download(url, dest):
    if not dest.exists():
        print("Downloading", url)
        urllib.request.urlretrieve(url, dest)
    return dest


IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def images_in(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in IMG_EXT)

## 1. Download datasets

In [ ]:
# PlantVillage: only the colour photos and the background-removed (segmented) versions
PV = DATA / "plantvillage"
if not (PV / "raw/color").exists():
    run(f"git clone --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git {PV}")
    run(f"cd {PV} && git sparse-checkout set raw/color raw/segmented")

# PlantDoc: real-world photos with leaf bounding boxes (used for both models)
PD = DATA / "plantdoc_od"
if not (PD / "TRAIN").exists():
    run(f"git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git {PD}")

# Imagenette: everyday non-plant photos for the "unknown" class and as backgrounds
IMAGENETTE = DATA / "imagenette2-320"
if not IMAGENETTE.exists():
    tgz = download("https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz", DATA / "imagenette.tgz")
    with tarfile.open(tgz) as t:
        t.extractall(DATA, filter="data")

# DTD textures (optional): hard negatives for the "unknown" class
DTD = DATA / "dtd"
try:
    if not (DTD / "images").exists():
        tgz = download("https://www.robots.ox.ac.uk/~vgg/data/dtd/download/dtd-r1.0.1.tar.gz", DATA / "dtd.tgz")
        with tarfile.open(tgz) as t:
            t.extractall(DATA, filter="data")
except Exception as e:
    print("Skipping DTD textures:", e)

print("Datasets ready.")

## 2. Build the classifier dataset

In [ ]:
import pandas as pd
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit, train_test_split

PV_COLOR, PV_SEG = PV / "raw/color", PV / "raw/segmented"
UNKNOWN = "unknown"
CLASSES = sorted(d.name for d in PV_COLOR.iterdir() if d.is_dir()) + [UNKNOWN]
CLS_IDX = {c: i for i, c in enumerate(CLASSES)}
print(len(CLASSES), "classes")


# PlantVillage: shots of the same physical leaf share an id ("Leaf 259", "Leaf 259.1"),
# so they are grouped and kept in the same split
def leaf_id(path):
    s = path.stem.split("___")[-1].replace("_final_masked", "")
    s = re.sub(r"\s*\(\d+\)$|\s+copy$", "", s)
    return re.sub(r"\.\d+$", "", s).strip().lower()


seg_lookup = {(p.parent.name, p.stem.replace("_final_masked", "")): str(p) for p in images_in(PV_SEG)}
pv_rows = []
for cdir in sorted(d for d in PV_COLOR.iterdir() if d.is_dir()):
    for p in images_in(cdir):
        pv_rows.append({
            "path": str(p), "label": CLS_IDX[cdir.name], "source": "pv",
            "group": f"{cdir.name}/{leaf_id(p)}", "seg": seg_lookup.get((cdir.name, p.stem), ""),
        })
pv = pd.DataFrame(pv_rows)
tr, va = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED).split(pv, groups=pv["group"]))
pv["split"] = "train"
pv.loc[pv.index[va], "split"] = "val"

# PlantDoc label -> PlantVillage class
PD_TO_PV = {
    "Apple Scab Leaf": "Apple___Apple_scab",
    "Apple leaf": "Apple___healthy",
    "Apple rust leaf": "Apple___Cedar_apple_rust",
    "Bell_pepper leaf": "Pepper,_bell___healthy",
    "Bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot",
    "Blueberry leaf": "Blueberry___healthy",
    "Cherry leaf": "Cherry_(including_sour)___healthy",
    "Corn Gray leaf spot": "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
    "Corn rust leaf": "Corn_(maize)___Common_rust_",
    "Peach leaf": "Peach___healthy",
    "Potato leaf": "Potato___healthy",
    "Potato leaf early blight": "Potato___Early_blight",
    "Potato leaf late blight": "Potato___Late_blight",
    "Raspberry leaf": "Raspberry___healthy",
    "Soyabean leaf": "Soybean___healthy",
    "Soybean leaf": "Soybean___healthy",
    "Squash Powdery mildew leaf": "Squash___Powdery_mildew",
    "Strawberry leaf": "Strawberry___healthy",
    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato leaf": "Tomato___healthy",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato mold leaf": "Tomato___Leaf_Mold",
    "Tomato two spotted spider mites leaf": "Tomato___Spider_mites Two-spotted_spider_mite",
    "grape leaf": "Grape___healthy",
    "grape leaf black rot": "Grape___Black_rot",
}


def norm(s):
    return " ".join(str(s).lower().replace("_", " ").split())


PD_MAP = {norm(k): v for k, v in PD_TO_PV.items() if v in CLS_IDX}


def read_boxes(row, W, H):
    sx, sy = W / float(row["width"]), H / float(row["height"])
    x1, y1, x2, y2 = (float(row[k]) for k in ("xmin", "ymin", "xmax", "ymax"))
    x1, x2 = max(0.0, x1 * sx), min(float(W), x2 * sx)
    y1, y2 = max(0.0, y1 * sy), min(float(H), y2 * sy)
    return x1, y1, x2, y2


# Real-world leaf crops cut from PlantDoc boxes, padded 10% like the backend pipeline
CROPS = DATA / "plantdoc_crops"


def build_crops(split_dir, csv_file, split_name):
    df = pd.read_csv(csv_file)
    lookup = {p.name: p for p in images_in(split_dir)}
    rows, skipped = [], 0
    for fname, g in df.groupby("filename"):
        src = lookup.get(fname)
        if src is None:
            continue
        try:
            img = Image.open(src).convert("RGB")
        except Exception:
            continue
        W, H = img.size
        for i, (_, r) in enumerate(g.iterrows()):
            target = PD_MAP.get(norm(r["class"]))
            try:
                x1, y1, x2, y2 = read_boxes(r, W, H)
            except (ValueError, ZeroDivisionError):
                target = None
            if target is None or x2 - x1 < 24 or y2 - y1 < 24:
                skipped += 1
                continue
            pw, ph = 0.1 * (x2 - x1), 0.1 * (y2 - y1)
            crop = img.crop((max(0, x1 - pw), max(0, y1 - ph), min(W, x2 + pw), min(H, y2 + ph)))
            crop.thumbnail((320, 320))
            out = CROPS / split_name / target / f"{Path(fname).stem[:60]}_{i}.jpg"
            out.parent.mkdir(parents=True, exist_ok=True)
            crop.save(out, quality=92)
            rows.append({"path": str(out), "label": CLS_IDX[target], "source": "real", "group": fname, "seg": ""})
    print(f"{split_name}: {len(rows)} crops, {skipped} boxes skipped")
    return pd.DataFrame(rows)


shutil.rmtree(CROPS, ignore_errors=True)
real = build_crops(PD / "TRAIN", PD / "train_labels.csv", "train")
tr, va = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED).split(real, groups=real["group"]))
real["split"] = "train"
real.loc[real.index[va], "split"] = "val"
real_test = build_crops(PD / "TEST", PD / "test_labels.csv", "test")
real_test["split"] = "test"

# "unknown" class: everyday objects + textures; "veined" textures are skipped because they are mostly leaves
neg = images_in(IMAGENETTE / "train")
random.shuffle(neg)
dtd = [p for p in images_in(DTD / "images") if p.parent.name != "veined"] if (DTD / "images").exists() else []
random.shuffle(dtd)
unk_paths = [str(p) for p in neg[:4000] + dtd[:1500]]
BG_PATHS = [str(p) for p in images_in(IMAGENETTE / "val")] + [str(p) for p in dtd[1500:]]

unk = pd.DataFrame({"path": unk_paths, "label": CLS_IDX[UNKNOWN], "source": "unk", "group": unk_paths, "seg": ""})
u_tr, u_rest = train_test_split(unk.index, test_size=0.2, random_state=SEED)
u_va, u_te = train_test_split(u_rest, test_size=0.5, random_state=SEED)
unk["split"] = "train"
unk.loc[u_va, "split"] = "val"
unk.loc[u_te, "split"] = "test"

# Your own photos (optional)
MY_PHOTOS = Path("/content/my_test_photos")
mine_rows = []
if MY_PHOTOS.exists():
    for cdir in MY_PHOTOS.iterdir():
        if cdir.is_dir() and cdir.name in CLS_IDX:
            for p in images_in(cdir):
                mine_rows.append({"path": str(p), "label": CLS_IDX[cdir.name], "source": "mine", "group": str(p), "seg": "", "split": "test"})
mine = pd.DataFrame(mine_rows, columns=pv.columns)

all_df = pd.concat([pv, real, real_test, unk, mine], ignore_index=True)
print(all_df.groupby(["split", "source"]).size().unstack(fill_value=0))

## 3. Augmentation and data loaders

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T
from tqdm.auto import tqdm

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.35, 1.0), ratio=(0.75, 1.33)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(30),
    T.RandomApply([T.ColorJitter(0.4, 0.4, 0.3, 0.05)], p=0.8),
    T.RandomApply([T.GaussianBlur(7, sigma=(0.1, 2.5))], p=0.3),
    T.RandomGrayscale(p=0.05),
    T.RandomAdjustSharpness(2, p=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.25, scale=(0.02, 0.12)),
])
# Same as the backend: plain resize to 224x224, ImageNet normalisation
eval_tf = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(MEAN, STD)])


# Pastes a background-removed PlantVillage leaf onto a random real photo,
# so the model stops relying on the grey lab background
def composite_on_background(seg_path):
    leaf = np.asarray(Image.open(seg_path).convert("RGB"))
    mask = (leaf.max(axis=2) > 20).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return Image.fromarray(leaf)
    filled = np.zeros_like(mask)
    cv2.drawContours(filled, [max(contours, key=cv2.contourArea)], -1, 1, thickness=-1)
    alpha = cv2.GaussianBlur(filled.astype(np.float32), (5, 5), 0)[..., None]
    h, w = mask.shape
    bg = T.RandomResizedCrop((h, w), scale=(0.2, 1.0))(Image.open(random.choice(BG_PATHS)).convert("RGB"))
    out = leaf * alpha + np.asarray(bg, dtype=np.float32) * (1 - alpha)
    return Image.fromarray(out.astype(np.uint8))


class LeafDataset(Dataset):
    def __init__(self, df, tf, composite_p=0.0):
        self.paths, self.labels, self.segs = df.path.tolist(), df.label.tolist(), df.seg.tolist()
        self.tf, self.p = tf, composite_p

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            if self.segs[i] and random.random() < self.p:
                img = composite_on_background(self.segs[i])
            else:
                img = Image.open(self.paths[i]).convert("RGB")
        except Exception:
            img = Image.open(self.paths[i]).convert("RGB")
        return self.tf(img), self.labels[i]


train_df = all_df[all_df.split == "train"].reset_index(drop=True)
val_df = all_df[all_df.split == "val"].reset_index(drop=True)
test_df = all_df[all_df.split == "test"].reset_index(drop=True)

# Each batch is ~50% PlantVillage, ~35% real-world, ~15% unknown, balanced across classes
SOURCE_SHARE = {"pv": 0.5, "real": 0.35, "unk": 0.15}
counts = train_df.groupby(["source", "label"]).size()
n_cls = train_df.groupby("source").label.nunique()
weights = [SOURCE_SHARE[s] / (n_cls[s] * counts[(s, l)]) for s, l in zip(train_df.source, train_df.label)]
sampler = WeightedRandomSampler(weights, num_samples=CLS_EPOCH_SIZE, replacement=True)

workers = os.cpu_count() or 2
loader_args = dict(batch_size=CLS_BATCH, num_workers=workers, pin_memory=True, persistent_workers=True)
train_loader = DataLoader(LeafDataset(train_df, train_tf, composite_p=0.5), sampler=sampler, drop_last=True, **loader_args)
val_loader = DataLoader(LeafDataset(val_df, eval_tf), shuffle=False, **loader_args)
test_loader = DataLoader(LeafDataset(test_df, eval_tf), shuffle=False, **loader_args)

# Preview of training images after augmentation
import matplotlib.pyplot as plt
xb, yb = next(iter(train_loader))
plt.figure(figsize=(14, 6))
for i in range(12):
    img = xb[i].permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)
    plt.subplot(2, 6, i + 1)
    plt.imshow(img.clip(0, 1))
    plt.title(CLASSES[yb[i]][:22], fontsize=7)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Train the classifier (head first, then full fine-tune)

In [ ]:
device = "cuda"
torch.manual_seed(SEED)
model = timm.create_model("mobilenetv4_conv_small.e1200_r224_in1k", pretrained=True, num_classes=len(CLASSES)).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler("cuda")
BEST_PATH = OUT / "best_classifier.pth"
history, best = [], {"score": -1.0}


def predict(loader):
    model.eval()
    out, labels = [], []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for x, y in loader:
            out.append(model(x.to(device, non_blocking=True)).float().cpu())
            labels.append(y)
    return torch.cat(out), torch.cat(labels)


def accuracy_by_source(logits, labels, sources):
    preds, src = logits.argmax(1).numpy(), np.asarray(sources)
    return {s: float((preds[src == s] == labels.numpy()[src == s]).mean() * 100) for s in np.unique(src)}


def run_stage(name, epochs, lr, train_backbone):
    for p in model.parameters():
        p.requires_grad = train_backbone
    for p in model.get_classifier().parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_loader), pct_start=0.1)
    for ep in range(1, epochs + 1):
        model.train()
        total, n = 0.0, 0
        for x, y in tqdm(train_loader, desc=f"{name} {ep}/{epochs}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            total, n = total + loss.item() * x.size(0), n + x.size(0)
        acc = accuracy_by_source(*predict(val_loader), val_df.source)
        # Model selection favours real-world accuracy
        score = 0.6 * acc.get("real", 0) + 0.2 * acc.get("pv", 0) + 0.2 * acc.get("unk", 0)
        history.append({"stage": name, "loss": total / n, **acc, "score": score})
        print(f"{name} {ep}: loss={total / n:.3f} real={acc.get('real', 0):.1f}% lab={acc.get('pv', 0):.1f}% unknown={acc.get('unk', 0):.1f}%")
        if score > best["score"]:
            best.update(score=score, **acc)
            torch.save(model.state_dict(), BEST_PATH)


run_stage("head", CLS_HEAD_EPOCHS, 1e-3, train_backbone=False)
run_stage("full", CLS_FULL_EPOCHS, 1e-4, train_backbone=True)
print("Best validation:", best)

## 5. Calibrate confidence and test on real-world photos

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

model.load_state_dict(torch.load(BEST_PATH))

# Temperature scaling: makes "70% confident" actually mean ~70% correct, so the backend threshold is meaningful
val_logits, val_labels = predict(val_loader)
log_t = torch.zeros(1, requires_grad=True)
lbfgs = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)


def closure():
    lbfgs.zero_grad()
    loss = nn.functional.cross_entropy(val_logits / log_t.exp(), val_labels)
    loss.backward()
    return loss


lbfgs.step(closure)
TEMPERATURE = float(log_t.exp())
print(f"Temperature: {TEMPERATURE:.3f}")

test_logits, test_labels = predict(test_loader)
probs = torch.softmax(test_logits / TEMPERATURE, 1)
conf, pred = probs.max(1)
conf, pred, y = conf.numpy(), pred.numpy(), test_labels.numpy()
src = test_df.source.to_numpy()
unk_id = CLS_IDX[UNKNOWN]

print("\nTest accuracy (never seen during training):")
for s, acc in accuracy_by_source(test_logits, test_labels, src).items():
    print(f"  {s:>5}: {acc:.1f}%  ({(src == s).sum()} images)")

# Threshold table: how often the app gives an answer, and how often that answer is right
plant = y != unk_id
rows = []
for t in np.arange(0.3, 0.95, 0.05):
    answered = (conf >= t) & (pred != unk_id)
    rows.append({
        "threshold": round(t, 2),
        "plant photos answered %": 100 * answered[plant].mean(),
        "accuracy when answered %": 100 * (pred[answered & plant] == y[answered & plant]).mean() if (answered & plant).any() else 0,
        "non-plant photos rejected %": 100 * (~answered[~plant]).mean() if (~plant).any() else 0,
    })
table = pd.DataFrame(rows).round(1)
print(table.to_string(index=False))
good = table[table["accuracy when answered %"] >= 85]
RECOMMENDED_THRESHOLD = float(good.threshold.iloc[0]) if len(good) else 0.6
print(f"\nRecommended CLASSIFIER_CONF_THRESHOLD: {RECOMMENDED_THRESHOLD}")

hist = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(hist.loss)
ax[0].set_title("Training loss")
for col, lbl in (("real", "Real-world"), ("pv", "Lab (PlantVillage)"), ("unk", "Unknown")):
    if col in hist:
        ax[1].plot(hist[col], label=lbl)
ax[1].set_title("Validation accuracy %")
ax[1].legend()
plt.tight_layout()
plt.savefig(FIGURES / "training_curves.png", dpi=200)
plt.show()

mask = np.isin(src, ["real", "unk"])
labels_present = sorted(set(y[mask]) | set(pred[mask]))
cm = confusion_matrix(y[mask], pred[mask], labels=labels_present, normalize="true")
plt.figure(figsize=(16, 14))
names = [CLASSES[i].replace("___", " / ")[:30] for i in labels_present]
sns.heatmap(cm, cmap="Greens", xticklabels=names, yticklabels=names, vmin=0, vmax=1)
plt.title("Real-world test confusion matrix")
plt.tight_layout()
plt.savefig(FIGURES / "confusion_matrix_real_world.png", dpi=200)
plt.show()

## 6. Export the classifier to ONNX

In [ ]:
import onnxruntime as ort


# Temperature is baked into the exported model so the backend needs no changes
class CalibratedModel(nn.Module):
    def __init__(self, m, t):
        super().__init__()
        self.m = m
        self.register_buffer("t", torch.tensor(t, dtype=torch.float32))

    def forward(self, x):
        return self.m(x) / self.t


export_model = CalibratedModel(model.float().cpu().eval(), TEMPERATURE).eval()
dummy = torch.randn(1, 3, 224, 224)
raw_onnx = OUT / "mobilenetv4_raw.onnx"
export_kwargs = dict(opset_version=17, input_names=["input"], output_names=["logits"])
try:
    torch.onnx.export(export_model, dummy, str(raw_onnx), dynamo=False, **export_kwargs)
except TypeError:
    torch.onnx.export(export_model, dummy, str(raw_onnx), **export_kwargs)

CLS_ONNX = EXPORT / "mobilenetv4_plant_disease.onnx"
run(f'onnxslim "{raw_onnx}" "{CLS_ONNX}"')

sess = ort.InferenceSession(str(CLS_ONNX), providers=["CPUExecutionProvider"])
diff = np.abs(sess.run(None, {sess.get_inputs()[0].name: dummy.numpy()})[0] - export_model(dummy).detach().numpy()).max()
print(f"ONNX vs PyTorch max difference: {diff:.6f}")
assert diff < 1e-3

## 7. Train the YOLO leaf detector

In [ ]:
YOLO_DIR = DATA / "yolo_leaf"
shutil.rmtree(YOLO_DIR, ignore_errors=True)
for s in ("train", "val", "test"):
    (YOLO_DIR / "images" / s).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / s).mkdir(parents=True, exist_ok=True)


# Converts PlantDoc CSV boxes into YOLO format; every box becomes class 0 "leaf"
# (the classifier decides the disease on each crop)
def convert(split_dir, csv_file, pick_split):
    df = pd.read_csv(csv_file)
    lookup = {p.name: p for p in images_in(split_dir)}
    n = 0
    for fname, g in df.groupby("filename"):
        src = lookup.get(fname)
        if src is None:
            continue
        try:
            img = Image.open(src).convert("RGB")
        except Exception:
            continue
        W, H = img.size
        lines = []
        for _, r in g.iterrows():
            try:
                x1, y1, x2, y2 = read_boxes(r, W, H)
            except (ValueError, ZeroDivisionError):
                continue
            if x2 - x1 < 4 or y2 - y1 < 4:
                continue
            lines.append(f"0 {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} {(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")
        if not lines:
            continue
        split = pick_split(fname)
        stem = f"{split}_{n:05d}"
        n += 1
        img.save(YOLO_DIR / "images" / split / f"{stem}.jpg", quality=95)
        (YOLO_DIR / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))
    return n


n_train = convert(PD / "TRAIN", PD / "train_labels.csv", lambda f: "val" if zlib.crc32(f.encode()) % 10 == 0 else "train")
n_test = convert(PD / "TEST", PD / "test_labels.csv", lambda f: "test")
print(f"YOLO images: {n_train} train+val, {n_test} test")

DATA_YAML = YOLO_DIR / "data.yaml"
DATA_YAML.write_text("\n".join([
    f"path: {YOLO_DIR}",
    "train: images/train",
    "val: images/val",
    "test: images/test",
    "names:",
    "  0: leaf",
]))

from ultralytics import YOLO

yolo = YOLO("yolo11n.pt")
yolo.train(
    data=str(DATA_YAML), epochs=YOLO_EPOCHS, imgsz=640, batch=16, patience=15,
    cos_lr=True, mosaic=1.0, mixup=0.1, seed=SEED,
    project=str(OUT / "yolo_runs"), name="yolo11n_leaf", exist_ok=True,
)
best_yolo = YOLO(str(yolo.trainer.best))
metrics = best_yolo.val(data=str(DATA_YAML), split="test")
print(f"Detector test mAP50: {metrics.box.map50:.3f}  mAP50-95: {metrics.box.map:.3f}")

yolo_onnx = best_yolo.export(format="onnx", imgsz=640, opset=17, simplify=True, dynamic=False)
shutil.copy(yolo_onnx, EXPORT / "yolo11n_leaf.onnx")
DETECTION_CLASSES = {str(k): v for k, v in best_yolo.names.items()}
print("Detector classes:", DETECTION_CLASSES)

## 8. Write classes.json and download everything

In [ ]:
def get_treatment_metadata(class_name):
    parts = class_name.split("___")
    species = parts[0].replace("_", " ").title()
    disease_raw = parts[1].replace("_", " ") if len(parts) > 1 else "Healthy"
    d = disease_raw.lower()
    if "healthy" in d:
        return species, "None", True, "None", "Optimal physiological vigor. Maintain standard irrigation and routine fertigation cycles."
    severity, treatment = "Moderate", "Apply targeted copper-based or systemic fungicide and isolate infected canopy leaves."
    if "rust" in d:
        treatment = "Deploy rust-tolerant hybrids. Apply preventative azoxystrobin/propiconazole upon early pustule emergence."
    elif "late blight" in d:
        severity, treatment = "Critical", "URGENT: Rogue out severely infected tissue. Spray translaminar oomycete fungicide (mefenoxam or fluopicolide)."
    elif "early blight" in d:
        treatment = "Prune lower foliage (12 inches) to prevent splash contamination. Apply chlorothalonil on a 7-10 day cycle."
    elif "scab" in d:
        treatment = "Spray captan or myclobutanil at green tip through petal fall. Clear leaf litter in autumn."
    elif "virus" in d:
        severity, treatment = "Critical", "Vector control: Eradicate whiteflies/aphids using imidacloprid or insect screens. Rogue infected plants immediately."
    elif "bacteria" in d:
        treatment = "Spray fixed copper mixed with mancozeb. Refrain from overhead irrigation; work only when canopy is dry."
    elif "mite" in d:
        severity, treatment = "High", "Release biological predators (Phytoseiulus persimilis) or apply horticultural neem oil/abamectin."
    return species, disease_raw.title(), False, severity, treatment


classification_classes = []
for idx, cname in enumerate(CLASSES):
    if cname == UNKNOWN:
        entry = ("Unknown", "Unknown", False, "None", "Not a supported plant leaf. Retake the photo of a single leaf in good light.")
    else:
        entry = get_treatment_metadata(cname)
    species, disease, is_healthy, severity, treatment = entry
    classification_classes.append({
        "id": idx, "class_name": cname, "species": species, "disease": disease,
        "is_healthy": is_healthy, "severity": severity, "treatment": treatment,
    })

with open(EXPORT / "classes.json", "w", encoding="utf-8") as f:
    json.dump({"detection_classes": DETECTION_CLASSES, "classification_classes": classification_classes}, f, indent=2)

(EXPORT / "README.txt").write_text(
    "Copy into the project:\n"
    "  yolo11n_leaf.onnx, mobilenetv4_plant_disease.onnx, classes.json -> apps/ai-service/app/models/\n"
    "  mobilenetv4_plant_disease.onnx, classes.json -> apps/frontend/public/models/\n"
    f"\nRecommended CLASSIFIER_CONF_THRESHOLD: {RECOMMENDED_THRESHOLD}\n"
    f"Best validation: {best}\n"
)
for fig in FIGURES.glob("*.png"):
    shutil.copy(fig, EXPORT / fig.name)

zip_path = shutil.make_archive("/content/plant_models_v2", "zip", EXPORT)
print("Saved to", EXPORT)
print((EXPORT / "README.txt").read_text())

from google.colab import files
files.download(zip_path)